## Introduction

What each part is doing:

- Google Colab: Colab is simply acting as the host for your Python runtime. It's running the Gradio script that builds the web interface. It serves the webpage you see and handles the communication between the user's browser and the external API. Colab provides zero GPU/CPU power for the actual model computation.
- Hugging Face: Hugging Face is providing two services here:
Model Repository: They host the model card (`deepseek-ai/DeepSeek-V3.1`), which contains the information needed to run the model (the architecture, the tokenizer, etc.).
- Authentication (the `accept_token`): The `gr.LoginButton` and `accept_token` parameter handle the OAuth login flow. This allows the Gradio app to use the user's Hugging Face token to authenticate with the inference provider. Hugging Face itself is not serving the model.
- Fireworks.ai (The Inference Provider): This is the most important part. When you set `provider="fireworks-ai"`, you are telling Gradio to send all inference requests to the Fireworks.ai API. They are the company that has the powerful GPUs (likely A100s or H100s) loaded with the `DeepSeek-V3.1` model. They receive the prompt from your Gradio app running on Colab, run it through their model on their servers, generate the response, and send it back.


## Sequence Diagram

In [12]:
import base64
import sys
import json
import zlib
from IPython.display import SVG, Image
import requests

mermaid_code = '''
sequenceDiagram
    participant User
    participant GradioUI as Gradio UI
    participant HF as Hugging Face Auth
    participant Fireworks as Fireworks AI
    participant DeepSeek as DeepSeek-V3.1 Model

    Note over GradioUI: App launches with sidebar,<br/>markdown content, and login button

    User->>GradioUI: Accesses the interface
    GradioUI->>User: Shows sidebar with login button

    User->>GradioUI: Clicks "Sign in" button
    GradioUI->>HF: Redirects to Hugging Face OAuth
    HF->>User: Shows authentication page
    User->>HF: Provides credentials
    HF->>GradioUI: Returns authentication token
    GradioUI->>User: Shows authenticated interface with chat

    Note over GradioUI: gr.load() loads the DeepSeek model<br/>with Fireworks AI as provider

    User->>GradioUI: Sends chat message
    GradioUI->>Fireworks: Forwards message with auth token
    Fireworks->>DeepSeek: Routes request to DeepSeek-V3.1
    DeepSeek->>Fireworks: Returns AI response
    Fireworks->>GradioUI: Returns formatted response
    GradioUI->>User: Displays AI response in chat interface
'''

def js_btoa(data):
    return base64.b64encode(data)

def pako_deflate(data):
    compress = zlib.compressobj(9, zlib.DEFLATED, 15, 8, zlib.Z_DEFAULT_STRATEGY)
    compressed_data = compress.compress(data)
    compressed_data += compress.flush()
    return compressed_data

def genPakoLink(graphMarkdown: str):
    jGraph = {"code": graphMarkdown, "mermaid": {"theme": "standard"}}
    byteStr = json.dumps(jGraph).encode('utf-8')
    deflated = pako_deflate(byteStr)
    dEncode = js_btoa(deflated)
    link_code = dEncode.decode('ascii')
    return link_code

mermaid_link = genPakoLink(mermaid_code)
print("mermaid.live link:")
print('http://mermaid.live/edit#pako:' + mermaid_link)

mermaid.live link:
http://mermaid.live/edit#pako:eNqNVMFum0AQ/ZURp0ZyUlW9ocqSVYs6h7RVLPfEZb07gZVhl84OQVWUf++sCRhCE5WLMcx78+a9WZ4S7Q0mKSS5C/i7Radxa1VBqs4dyNUoYqttoxzDISAtn34jZaw/3IIKL/dwuF2W7bJYsGuLwroCMqURNi2Xy8LMEnaeTiHWX/5s/sG5RWz2iKdYOdxf//p88wnuZKoqdz3ku2cE/4g0ak1h0zRQqdbpEgN0lksI1uBR0erLkT6ua0Un4zsH2jtGxytQzkDlRTwcW2bvBvJoyvV6PWHWGkMQVi4RrKDpQYbti4cqAURcCvvSd2Fo3ev4ryZfK6vFlDzZ28JJlzwZEa/67LIU7tGIj5pFk59H8OOSwS57pUrJKxndasXWOzG+wJmayPyT/KOID6AJTSxWVZjQXQTfI7fkFqTsT+je9WYCQHPxs/dKl4rfS7mgm8or8+EK4k8fybg0ddyRc9pnrumqxYVq+tHozQz26IQySoBaEh/tmQwycqaQeeoUCeClth8gTjc1YQQIeBAq5vmWxWOKBzRwDHG27T10fDRrO/guMxGGxsspX3ZaxvTgqVYcHZ+jFiFtbWgq9WfWQFLqbZlsf7KCpEYhtUa+Nk+JJFGfvzuB5WSJMcnz819kEZAx


In [3]:
# acquires token from google colab secrets

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [10]:
# sets token as environment variable

import os
os.environ['HF_TOKEN'] = HF_TOKEN

Launch the app

In [11]:
import gradio as gr

with gr.Blocks(fill_height=True) as demo:
    with gr.Sidebar():
        gr.Markdown("# Inference Provider")
        gr.Markdown("This Space showcases the deepseek-ai/DeepSeek-V3.1 model, served by the fireworks-ai API. Sign in with your Hugging Face account to use this API.")
        button = gr.LoginButton("Sign in")
    gr.load("models/deepseek-ai/DeepSeek-V3.1", accept_token=button, provider="fireworks-ai")

demo.launch()

/usr/local/lib/python3.12/dist-packages/gradio/oauth.py:162: UserWarning: Gradio does not support OAuth features outside of a Space environment. To help you debug your app locally, the login and logout buttons are mocked with your profile. To make it work, your machine must be logged in to Huggingface.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a00efda27f6dd0e793.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
